## CCF & SHAP

### Cell 1: 库导入和路径设置

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as xgb
import os

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

PROCESSED_DIR = '../data/processed/'
FIGURE_DIR = '../results/figures/'
os.makedirs(FIGURE_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(PROCESSED_DIR, 'train_set.csv'), index_col='datetime', parse_dates=True)
print("Train shape:", df.shape)

### Cell 2: 交叉相关性 (CCF) 分析与滞后性检验

In [ ]:
def plot_cross_correlation(series_target, series_feature, feature_name, max_lag=24):
    ccf_values = []
    lags = range(0, max_lag + 1)
    
    for lag in lags:
        temp_df = pd.concat([series_target, series_feature.shift(lag)], axis=1).dropna()
        if len(temp_df) > 0:
            corr = temp_df.iloc[:, 0].corr(temp_df.iloc[:, 1])
        else:
            corr = 0
        ccf_values.append(corr)
    
    plt.figure(figsize=(10, 5), dpi=150)
    plt.bar(lags, ccf_values, color='skyblue', edgecolor='black')
    plt.title(f'CCF between PM2.5 and {feature_name}', fontsize=14)
    plt.xlabel('Delay time (h)', fontsize=12)
    plt.ylabel('Pearson correlation coefficient', fontsize=12)
    plt.xticks(lags)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    max_lag_idx = np.argmax(np.abs(ccf_values))
    plt.axvline(x=max_lag_idx, color='red', linestyle='--', label=f'Maximum impact lag: {max_lag_idx} hours')
    plt.legend()
    
    plt.savefig(os.path.join(FIGURE_DIR, f'ccf_{feature_name}_lag.png'), bbox_inches='tight')
    plt.show()
    return max_lag_idx

plot_cross_correlation(df['PM2.5'], df['temperature'], 'Temperature')
plot_cross_correlation(df['PM2.5'], df['humidity'], 'Humidity')
plot_cross_correlation(df['PM2.5'], df['Wind_Speed(m/s)'], 'Wind_Spee')

### Cell 3: 构建用于 SHAP 分析的特征工程

In [ ]:
def build_lag_features(data, target_col, lags):
    df_lag = data.copy()
    feature_cols = []
    
    meteorological_features = [
        'temperature', 'humidity', 'Wind_Speed(m/s)', 
        'PBL_Height(m)', 'Precipitation(m)', 'Solar_Radiation(J/m2)'
    ]
    
    # 添加滞后特征
    for col in meteorological_features:
        if col in df_lag.columns:
            for lag in lags:
                new_col = f'{col}_lag_{lag}'
                df_lag[new_col] = df_lag[col].shift(lag)
                feature_cols.append(new_col)
            
    # 加入时间特征
    df_lag['hour'] = df_lag.index.hour
    feature_cols.append('hour')
    
    df_lag = df_lag.dropna()
    return df_lag[feature_cols], df_lag[target_col]

# 选取 1, 6, 12 小时前的全量气象特征作为输入，预测当前的 PM2.5
X, y = build_lag_features(df, 'PM2.5', lags=[1, 6, 12])
print("\nShape of SHAP Feature matrix X:", X.shape)

### Cell 4: 训练 XGBoost 模型与 SHAP 特征归因分析

In [ ]:
# Random seed
model = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42)
model.fit(X, y)

print("Calculating SHAP...")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

# 绘制 SHAP 摘要图 (Summary Plot)
plt.figure(figsize=(12, 8), dpi=150)
shap.summary_plot(shap_values, X, show=False)
plt.title('SHAP attribution analysis of PM2.5 concentration', fontsize=16, y=1.05)
plt.savefig(os.path.join(FIGURE_DIR, 'shap_summary_extended.png'), bbox_inches='tight')
plt.show()

print("CCF and SHAP Analysis Pipeline Ready.")